# Xây dựng Knowledge Base và đánh giá QA retrieval

Audit XIV/XVI/XVII + GIR + TT85 → metadata namespace/chunk_type → BGE-M3 dense/sparse → PostgreSQL/pgvector → graph và kiểm chứng. Hai namespace dùng cùng vector store. Biểu mẫu TT85 chỉ nằm trong danh mục template, không embedding. Logic dùng chung trong rag/; kết quả giữ tại output notebook. Hiệu lực pháp lý là metadata nguồn local, chưa xác minh độc lập trong workflow này.


In [ ]:
import os
import sys
from pathlib import Path

candidates = [Path(os.environ["AIAGENT_ROOT"])] if "AIAGENT_ROOT" in os.environ else []
for parent in [Path.cwd(), *Path.cwd().parents]:
    candidates.extend([parent, parent / "apps/aiAgent", parent / "frappe-bench/apps/aiAgent"])
ROOT = next((p.resolve() for p in candidates if (p / "rag/data.py").is_file()), None)
if ROOT is None:
    raise RuntimeError("Set AIAGENT_ROOT to the absolute aiAgent folder")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from rag.data import config  # noqa: E402 - project root must be resolved first

cfg = config()
print("Workspace:", ROOT)

In [ ]:
import time

import numpy as np
import pandas as pd
from IPython.display import display
from IPython.utils.io import capture_output

from rag.data import audit, build_corpus
from rag.db import connect, counts, ingest, migrate
from rag.embedding import Embedder, load_embeddings

## Đối chiếu dữ liệu nguồn sau chuẩn hoá

Kiểm tra schema v2 và văn bản tìm kiếm tiếng Việt theo từng bản ghi. Các sửa OCR lịch sử đã nằm trong `data/normalized`; bước này chỉ đọc dữ liệu và không ghi lại nguồn.


In [ ]:
from rag.data import source_files, tariff_bundle
from rag.sources import source_manifest

registered = source_manifest(ROOT)
data = source_files()
for source in registered["tariffs"]:
    records, nodes, edges = tariff_bundle(source, data)
    assert len({r["id"] for r in records}) == len(records)
    assert all(r["locator"] and r["source_text"] for r in records)
    print(source["records"], len(records), len(nodes), len(edges), "source adapter: PASS")


## 1. Audit và truy vết nguồn
TT31 ingest được nối với master; GIR combined và TT85 source là allowlist. data/verified/hs/dev_test gồm Dev và Test được dùng kiểm coverage mã, không đưa vào corpus.

In [ ]:
from rag.data import NORMALIZED, SOURCES

audit_result = audit()
display(pd.DataFrame(audit_result["quality"]).T[["count", "duplicate_wording"]])
display(
    pd.DataFrame(
        [
            {
                "scope_chapters": audit_result["scope_count"],
                "graph_edges": audit_result["graph_edges"],
                "broken_endpoints": len(audit_result["broken_endpoints"]),
                "gold_codes_missing": len(audit_result["gold_missing"]),
                "TT85_SHA256_match": audit_result["tt85_source_hash_matches"],
                "legal_verification": audit_result["legal_verification"],
            }
        ]
    )
)



display(pd.DataFrame([
    {"records": source["records"], "edges": source.get("edges"), "nodes": source.get("nodes")}
    for source in registered["tariffs"]
]))
print("Only manifest-registered sources enter ingestion.")


## 2. Corpus và metadata thống nhất
GIR giữ source atomic và context view; TT85 giữ điều khoản, loại 5 forms khỏi RAG. Structural/context/footnote chunks giữ nguyên vai trò và chunk_type=null khi không thuộc 7 loại statutory/tariff đã chốt, tránh đổi tên footnote thành điều luật. Scope index ngoài 3 Sections chỉ hỗ trợ nhận biết phạm vi.


In [ ]:
bundle = build_corpus()
records = pd.DataFrame(bundle["records"])
display(
    records.groupby(["namespace", "dataset", "section", "role"], dropna=False)
    .size()
    .rename("records")
    .reset_index()
)
display(
    records.groupby(["chunk_type", "level"], dropna=False).size().rename("records").reset_index()
)
print("Snapshot:", bundle["snapshot"], "Corpus:", len(records))
assert len(records) == bundle["manifest"]["count"]
assert records["id"].is_unique
assert not records["id"].str.contains("TT85-FORM-").any()
assert set(records["namespace"]) == {"hs_classification", "legal_classification"}

node_ids = {n["id"] for n in bundle["nodes"]}
assert all(e["source"] in node_ids and e["target"] in node_ids for e in bundle["edges"])
assert all(r["locator"] for r in bundle["records"])
print("Source hashes:", bundle["manifest"]["source_hashes"])
print("Graph:", len(bundle["nodes"]), "nodes /", len(bundle["edges"]), "edges")


In [ ]:
from rag.data import NORMALIZED, read

examples = [
    r
    for r in bundle["records"]
    if r["id"] in ["gir:GIR_3_NOTE_038", "gir:GIR_5_NOTE_016", "tt85:TT85-A9-3.a.1"]
]
display(
    pd.DataFrame(
        [
            {
                "id": r["id"],
                "source_text": r["source_text"],
                "embedding_view": r["embedding_text"],
                "locator": r["locator"],
            }
            for r in examples
        ]
    )
)

forms = read(NORMALIZED / "TT85/tt85_forms_source.json")
display(
    pd.DataFrame(
        [
            {
                "form": f["form_no"],
                "code": f["code"],
                "title": f["title"],
                "fields": len(f["fields"]),
            }
            for f in forms
        ]
    )
)
print("5 forms are a template registry, not RAG chunks; Word generation is not implemented.")
assert len(forms) == 5

## 3. BGE-M3 dense + sparse
Pin model/tokenizer commit. Corpus input không được truncate im lặng. Checkpoint theo batch/input hash giúp tái chạy nhanh. CPU FP32 và sparse là cấu hình đã chọn cho experiment; dense là 1.024 chiều, sparse là token weights của model.

In [ ]:
embedder = Embedder(cfg)
started = time.perf_counter()
with capture_output() as embedding_progress:
    embedding_manifest = embedder.corpus(bundle)
manifest, vectors, sparse = load_embeddings(bundle)
display(
    pd.DataFrame(
        [
            {
                "records": len(vectors),
                "dimension": vectors.shape[1],
                "max_input_tokens": manifest["max_tokens"],
                "sparse_nonzero": sum(len(w) for w in sparse),
                "elapsed_seconds": round(time.perf_counter() - started, 2),
                "model_revision": manifest["profile"]["revision"],
                "profile": manifest["profile_id"],
            }
        ]
    )
)
np.testing.assert_allclose(np.linalg.norm(vectors, axis=1), 1, atol=1e-5)
assert len(sparse) == len(vectors)
assert np.isfinite(vectors).all()

## 4. PostgreSQL: nạp snapshot đã kiểm chứng

Mặc định giữ snapshot cũ và kích hoạt snapshot mới trong transaction. `FRESH_REBUILD=True` vẫn là lựa chọn xóa schema RAG khi người vận hành chủ động chọn; không cần để thêm chương. Embeddings không đổi được tái sử dụng theo hash.


In [ ]:
conn = connect()
migrate(conn)
display(pd.DataFrame([conn.execute("SELECT current_database() AS database, version() AS server").fetchone()]))
APPLY = True
FRESH_REBUILD = False
RAG_TABLES = (
    "retrieval_results", "evaluation_runs", "active", "sparse_postings",
    "embeddings", "graph_edges", "graph_nodes", "chunks", "snapshots", "profiles",
)
actual_tables = {row["table_name"] for row in conn.execute(
    "SELECT table_name FROM information_schema.tables WHERE table_schema='rag' AND table_type='BASE TABLE'"
).fetchall()}
assert actual_tables == set(RAG_TABLES), (actual_tables, RAG_TABLES)
before_rebuild = {table: conn.execute(f"SELECT count(*) AS n FROM rag.{table}").fetchone()["n"]
                  for table in RAG_TABLES}
print("RAG database before fresh rebuild:", before_rebuild)
print("Apply:", APPLY, "fresh rebuild:", FRESH_REBUILD)


In [ ]:
if APPLY and FRESH_REBUILD:
    # All statements run on the isolated RAG schema; no CASCADE into other schemas.
    with conn.transaction():
        conn.execute("SELECT pg_advisory_xact_lock(20260926)")
        conn.execute("TRUNCATE TABLE " + ", ".join(f"rag.{table}" for table in RAG_TABLES))
        assert all(conn.execute(f"SELECT count(*) AS n FROM rag.{table}").fetchone()["n"] == 0
                   for table in RAG_TABLES)
        print("Old RAG tables cleared inside transaction.")
        ingestion_result = ingest(conn, bundle, manifest, vectors, sparse, apply=True)
        assert conn.execute("SELECT count(*) AS n FROM rag.snapshots").fetchone()["n"] == 1
        assert conn.execute("SELECT count(*) AS n FROM rag.chunks").fetchone()["n"] == len(bundle["records"])
        assert conn.execute("SELECT snapshot FROM rag.active").fetchone()["snapshot"] == bundle["snapshot"]
else:
    ingestion_result = ingest(conn, bundle, manifest, vectors, sparse, apply=APPLY)
display(pd.DataFrame([ingestion_result.get("counts", ingestion_result.get("expected"))]))
if APPLY:
    after = counts(conn, bundle["snapshot"], manifest["profile_id"])
    assert after == ingestion_result["counts"]
    print("Fresh snapshot and counts: PASS")
    display(pd.DataFrame([conn.execute("SELECT * FROM rag.active").fetchone()]))


## 5. Kiểm chứng cuối
Counts, FK integrity, vector norms và sparse coverage đều được kiểm trực tiếp. Phần QA phía dưới và notebook 02 dùng snapshot/profile active này; không cần giữ kernel hiện tại.

In [ ]:
if APPLY:
    check = conn.execute(
        """SELECT count(*) AS sparse_postings FROM rag.sparse_postings
        WHERE snapshot=%s AND profile=%s""",
        (bundle["snapshot"], manifest["profile_id"]),
    ).fetchone()
    assert check["sparse_postings"] == sum(len(w) for w in sparse)
    print(
        "Knowledge base ready:", {"snapshot": bundle["snapshot"], "records": len(vectors), **check}
    )
conn.close()

## Definition graph audit for retrieval
The active knowledge base already contains Notes and definition edges. Derive cited search views from that active snapshot; do not re-ingest gold or change atomic source wording. Search views are built in memory for BM25 and optional evidence-aware reranking. Existing BGE dense/sparse vectors remain valid. Notebook 02 benchmarks and freezes this retrieval policy.

In [ ]:
import pandas as pd
from IPython.display import display

from rag.db import connect
from rag.retrieval import Retriever

with connect() as audit_connection:
    definition_audit = Retriever(audit_connection)
    definition_views = definition_audit.definition_views()
    active_before = audit_connection.execute("SELECT * FROM rag.active").fetchone()
    display(pd.DataFrame([{"section": view["record"]["section"],
        "heading": view["record"]["heading"], "ahtn8": view["record"]["code"],
        "source_ids": [x["id"] for x in view["citations"]],
        "graph_paths": view["paths"]} for view in definition_views.values()]))
    assert all(view["record"]["source_text"] == definition_audit.exact(view["record"]["code"])[0]["source_text"]
        for view in definition_views.values())
    assert audit_connection.execute("SELECT * FROM rag.active").fetchone() == active_before
    print("Source-backed definition views:", len(definition_views), "Active snapshot:", active_before)


## QA retrieval trên cùng Knowledge Base

Phần này được gộp từ notebook 03. Nó đo truy xuất căn cứ cho GIR, Chú giải, dòng biểu thuế và TT85 trên **cùng snapshot**; riêng TT85 thuộc namespace `legal_classification`. Chỉ tập QA dev chọn trọng số/chính sách; QA test dùng để đánh giá. Kết quả và từng câu nằm ngay trong output bên dưới. Chính sách HS được giữ khi cập nhật khóa `qa`.


In [ ]:
import time

import pandas as pd
from IPython.display import display
from IPython.utils.io import capture_output

from rag.data import config, digest, load_corpus, read, write_json
from rag.db import connect
from rag.embedding import Embedder
from rag.evaluation import (
    CONFIGURATIONS,
    QA_CHUNK_TYPES,
    branch_cache,
    discover_qa,
    qa_direct_coverage,
    qa_objective,
    sweep_qa_weights,
)
from rag.retrieval import Retriever

pd.set_option("display.max_colwidth", 140)

bundle = load_corpus()
conn = connect()
with capture_output():
    embedder = Embedder(config())
runtime = Retriever(conn, embedder)
assert runtime.snapshot == bundle["snapshot"]
print("Active snapshot:", runtime.snapshot, "profile:", runtime.profile)
print("Corpus records:", len(bundle["records"]))

## 1. Kiểm tra dev và nguồn

Chỉ đọc `data/verified/qa/dev_test/LEGAL_RETRIEVAL_DEV.jsonl` ở bước chọn cấu hình. Mapping gold được dùng để **chấm điểm**, không đi vào query hay cơ sở tri thức.

In [ ]:
qa_dev = discover_qa(bundle, split="dev")
assert len(qa_dev) == 520
assert len({c["case_family_id"] for c in qa_dev}) == len(qa_dev)
assert all(c["canonical_groups"] for c in qa_dev)
dev_hash = digest(qa_dev)
source_by_id = {r["id"]: r for r in bundle["records"]}
dev_rows = []
for case in qa_dev:
    primary = [g["any_of"][0] for g in case["canonical_groups"]]
    dev_rows.append(
        {
            "case_type": case.get("case_type", ""),
            "section_scope": ",".join(case.get("section_scope", [])),
            "primary_tariff": any(source_by_id[s]["chunk_type"] == "tariff_entry" for s in primary),
            "primary_count": len(primary),
        }
    )
print("Dev:", len(qa_dev), "unique queries:", len({c["query"].casefold() for c in qa_dev}))
display(
    pd.DataFrame(dev_rows)
    .groupby(["case_type", "primary_tariff"])
    .size()
    .rename("count")
    .reset_index()
)
print("Dev hash:", dev_hash)

## 2. Chọn retriever và trọng số trên dev

Thử các họ Dense/Sparse/BM25/FTS/Exact và Weighted RRF như notebook 02. Chọn theo tỷ lệ **đủ mọi căn cứ primary trong Top-10**, rồi Recall và MRR. Đây là chọn retrieval policy, không fine-tune BGE-M3.

In [ ]:
with capture_output():
    dev_cache = branch_cache(
        runtime,
        qa_dev,
        intent="knowledge",
        pool=100,
        filters={"chunk_types": QA_CHUNK_TYPES},
        mask_questioned_codes=True,
    )
family_best = []
for family in CONFIGURATIONS:
    family_best.append(sweep_qa_weights(dev_cache, family)[0])
base_choice = max(family_best, key=qa_objective)
base_policy = {
    "configuration": base_choice["configuration"],
    "weights": base_choice["weights"],
    "k": base_choice["k"],
    "pool": 100,
    "reranker": "cross_encoder",
    "rerank_pool": 30,
    "rank_weight": 0.75,
    "graph_before_rerank": False,
    "query_split_weight": 0.0,
}
old_policy = {
    "configuration": "C11",
    "weights": {"dense": 0.30, "sparse": 0.05, "bm25": 0.30, "fts": 0.05, "exact": 0.30},
    "k": 5,
    "pool": 100,
    "reranker": "cross_encoder",
    "rerank_pool": 30,
    "rank_weight": 0.75,
    "graph_before_rerank": False,
    "query_split_weight": 0.0,
}
display(
    pd.DataFrame(family_best)[
        [
            "configuration",
            "weights",
            "k",
            "all_groups@5",
            "all_groups@10",
            "group_recall@10",
            "mrr@10",
        ]
    ]
)
print("Dev-selected base:", base_policy)
print("Previous snapshot baseline is evaluated as a transfer, not assumed optimal.")

## 3. Ablation trên 520 dev

Giữ cùng trọng số đã chọn và thay vị trí graph, kích thước pool, tỷ lệ giữ ứng viên trực tiếp, mức pha thứ hạng gốc và truy vấn bỏ mã nghi vấn. Cả heading/Notes mới đều phải là chunks có văn bản nguồn và locator. Latency bên dưới đo trên mô hình đã nạp, cache reranker có thể làm lần sau nhanh hơn; không dùng latency này để chọn chất lượng.

In [ ]:
variants = {
    "baseline_old_snapshot": old_policy,
    "direct_30": base_policy,
    "direct_50": {**base_policy, "rerank_pool": 50},
    "graph_30_70": {
        **base_policy,
        "graph_before_rerank": True,
        "rerank_pool": 30,
        "direct_fraction": 0.70,
    },
    "graph_50_50": {
        **base_policy,
        "graph_before_rerank": True,
        "rerank_pool": 50,
        "direct_fraction": 0.50,
    },
    "graph_50_70": {
        **base_policy,
        "graph_before_rerank": True,
        "rerank_pool": 50,
        "direct_fraction": 0.70,
    },
    "graph_50_85": {
        **base_policy,
        "graph_before_rerank": True,
        "rerank_pool": 50,
        "direct_fraction": 0.85,
    },
    "graph_100_70": {
        **base_policy,
        "graph_before_rerank": True,
        "rerank_pool": 100,
        "direct_fraction": 0.70,
    },
    "graph_50_pure": {
        **base_policy,
        "graph_before_rerank": True,
        "rerank_pool": 50,
        "direct_fraction": 0.70,
        "rank_weight": 1.0,
    },
    "graph_50_neutral_25": {
        **base_policy,
        "graph_before_rerank": True,
        "rerank_pool": 50,
        "direct_fraction": 0.70,
        "query_split_weight": 0.25,
    },
    "graph_50_neutral_50": {
        **base_policy,
        "graph_before_rerank": True,
        "rerank_pool": 50,
        "direct_fraction": 0.70,
        "query_split_weight": 0.50,
    },
}

with capture_output():
    runtime.qa_search(qa_dev[0]["query"], policy_override=base_policy, include_package=False)
dev_experiments = []
for name, policy in variants.items():
    started = time.perf_counter()
    for case in qa_dev:
        with capture_output():
            result = runtime.qa_search(case["query"], policy_override=policy, include_package=False)
        metric = qa_direct_coverage(case, result["matches"])
        required = {g["any_of"][0] for g in case["canonical_groups"]}
        pool_ids = set(result["pre_rerank_ids"])
        dev_experiments.append(
            {
                "variant": name,
                "case": case["id"],
                "case_type": case.get("case_type", ""),
                "section_scope": ",".join(case.get("section_scope", [])),
                "pool_complete": required <= pool_ids,
                "pool_recall": len(required & pool_ids) / len(required),
                "all_groups@5": metric["all_groups@5"],
                "all_groups@10": metric["all_groups@10"],
                "group_recall@10": metric["group_recall@10"],
                "mrr@10": metric["mrr@10"],
                "elapsed_ms": result["elapsed_ms"],
                "query_split_used": result["query_split_used"],
            }
        )
    print(name, "seconds:", round(time.perf_counter() - started, 1), flush=True)
dev_frame = pd.DataFrame(dev_experiments)
dev_summary = (
    dev_frame.groupby("variant")
    .agg(
        count=("case", "count"),
        pool_complete=("pool_complete", "mean"),
        pool_recall=("pool_recall", "mean"),
        **{
            "all_groups@5": ("all_groups@5", "mean"),
            "all_groups@10": ("all_groups@10", "mean"),
            "group_recall@10": ("group_recall@10", "mean"),
            "mrr@10": ("mrr@10", "mean"),
        },
        median_ms=("elapsed_ms", "median"),
        split_queries=("query_split_used", "sum"),
    )
    .reset_index()
)
display(dev_summary.sort_values(["all_groups@10", "group_recall@10", "mrr@10"], ascending=False))
display(
    dev_frame.groupby(["variant", "case_type"])["all_groups@10"]
    .agg(["mean", "count"])
    .reset_index()
)


## 4. Đóng băng QA policy trước khi mở test

Lấy phương án tốt nhất trên dev trong phạm vi đã thử. Nếu graph không thắng, nó vẫn được báo trong bảng test để chẩn đoán, nhưng không được âm thầm đưa vào runtime.

In [ ]:
candidate_names = list(variants)
new_candidate_names = [name for name in variants if name != "baseline_old_snapshot"]
quality = {name: dev_summary.set_index("variant").loc[name] for name in candidate_names}
chosen_name = max(
    candidate_names,
    key=lambda name: (
        quality[name]["all_groups@10"],
        quality[name]["group_recall@10"],
        quality[name]["all_groups@5"],
        quality[name]["mrr@10"],
    ),
)
selected_policy = {
    **variants[chosen_name],
    "selection_status": "QA_520_DEV_ONLY",
    "dev_sha256": dev_hash,
    "dev_count": len(qa_dev),
    "tested_variants": len(variants),
    "selected_variant": chosen_name,
    "objective": ["all_groups@10", "group_recall@10", "all_groups@5", "mrr@10"],
    "candidate_types": QA_CHUNK_TYPES,
    "limitations": "Best among tested policies on supplied dev; no model fine-tuning.",
}
hs_selection = read(ROOT / "config/selected_retrieval.json")
assert hs_selection["profile"] == runtime.profile
if hs_selection["snapshot"] != runtime.snapshot:
    # Bind the frozen QA configuration to the new source snapshot; HS uses the shared policy file.
    hs_selection = {**hs_selection, "snapshot": runtime.snapshot,
        "selection_scope": "SHARED_PRODUCT_POLICY; QA_DEV_SELECTION_ONLY"}
assert hs_selection["snapshot"] == runtime.snapshot
selected_config = {**hs_selection, "qa": selected_policy}
assert {k: v for k, v in selected_config.items() if k != "qa"} == {
    k: v for k, v in hs_selection.items() if k != "qa"
}
write_json(ROOT / "config/selected_retrieval.json", selected_config)
frozen_hash = digest(selected_config)
print("Frozen before test:", chosen_name, selected_policy)
print("Configuration digest:", frozen_hash)

## 5. Test 130 câu sau khi đóng băng

So sánh cách cũ, chính sách tìm trực tiếp, graph tốt nhất trên dev, và policy đã chọn. không dùng test để đổi trọng số hay chọn lại policy.

In [ ]:
assert digest(read(ROOT / "config/selected_retrieval.json")) == frozen_hash
qa_test = discover_qa(bundle, split="test")
assert len(qa_test) == 130
assert not ({c["case_family_id"] for c in qa_dev} & {c["case_family_id"] for c in qa_test})
dev_best_direct = max(
    (n for n in new_candidate_names if not variants[n]["graph_before_rerank"]),
    key=lambda n: (
        quality[n]["all_groups@10"],
        quality[n]["group_recall@10"],
        quality[n]["mrr@10"],
    ),
)
dev_best_graph = max(
    (n for n in new_candidate_names if variants[n]["graph_before_rerank"]),
    key=lambda n: (
        quality[n]["all_groups@10"],
        quality[n]["group_recall@10"],
        quality[n]["mrr@10"],
    ),
)
test_variants = dict.fromkeys(
    ["baseline_old_snapshot", dev_best_direct, dev_best_graph, chosen_name]
)
test_rows = []
for name in test_variants:
    policy = variants[name]
    started = time.perf_counter()
    for case in qa_test:
        with capture_output():
            result = runtime.qa_search(case["query"], policy_override=policy, include_package=True)
        metric = qa_direct_coverage(case, result["matches"])
        required = {g["any_of"][0] for g in case["canonical_groups"]}
        pool_ids = set(result["pre_rerank_ids"])
        package_ids = {r["id"] for r in result["package"]}
        test_rows.append(
            {
                "variant": name,
                "case": case["id"],
                "query": case["query"],
                "source_group": "old_50_diagnostic"
                if case.get("legacy_original_index") is not None
                else "new_100",
                "case_type": case.get("case_type", ""),
                "section_scope": ",".join(case.get("section_scope", [])),
                "pool_complete": required <= pool_ids,
                "pool_recall": len(required & pool_ids) / len(required),
                "all_groups@5": metric["all_groups@5"],
                "all_groups@10": metric["all_groups@10"],
                "group_recall@10": metric["group_recall@10"],
                "mrr@10": metric["mrr@10"],
                "package_complete": required <= package_ids,
                "package_recall": len(required & package_ids) / len(required),
                "missing_primary": sorted(required - {r["id"] for r in result["matches"]}),
                "returned_top10": [r["id"] for r in result["matches"]],
                "elapsed_ms": result["elapsed_ms"],
                "query_split_used": result["query_split_used"],
            }
        )
    print("TEST", name, "seconds:", round(time.perf_counter() - started, 1), flush=True)
test_frame = pd.DataFrame(test_rows)
columns = [
    "pool_complete",
    "pool_recall",
    "all_groups@5",
    "all_groups@10",
    "group_recall@10",
    "mrr@10",
    "package_complete",
    "package_recall",
]
test_summary = (
    test_frame.groupby("variant")[columns]
    .mean()
    .join(test_frame.groupby("variant").elapsed_ms.agg(["median", lambda s: s.quantile(0.95)]))
)
test_summary.columns = columns + ["latency_p50_ms", "latency_p95_ms"]
display(test_summary)
display(test_frame.groupby(["source_group", "variant"])[columns].mean())
display(
    test_frame.groupby(["case_type", "variant"])[
        ["all_groups@10", "group_recall@10", "package_complete"]
    ].mean()
)
print(
    "Frozen policy digest unchanged:",
    digest(read(ROOT / "config/selected_retrieval.json")) == frozen_hash,
)

In [ ]:
base_test = test_frame[test_frame.variant == dev_best_direct].set_index("case")
selected_test = test_frame[test_frame.variant == chosen_name].set_index("case")
comparison = selected_test[
    ["query", "source_group", "case_type", "section_scope", "missing_primary", "returned_top10"]
].copy()
comparison["direct_all@10"] = base_test["all_groups@10"]
comparison["selected_all@10"] = selected_test["all_groups@10"]
comparison["change"] = comparison["selected_all@10"].astype(int) - comparison[
    "direct_all@10"
].astype(int)
print(
    "Gains:",
    (comparison.change > 0).sum(),
    "losses:",
    (comparison.change < 0).sum(),
    "unchanged:",
    (comparison.change == 0).sum(),
)
display(comparison.loc[comparison.change != 0].head(30))
display(
    test_summary[["pool_complete", "all_groups@10", "package_complete"]]
    .style.format("{:.1%}")
    .bar(axis=0, color="#87a9d8", vmin=0, vmax=1)
    .set_caption("QA 150 test: đủ căn cứ trước rerank / Top-10 / gói căn cứ")
)


## 6. Runtime và kiểm tra

Đối chiếu runtime dùng cấu hình đã đóng băng với đường thử nghiệm. Kiểm tra chính sách HS không bị thay đổi khi thêm khóa QA; kết quả HS và QA thuộc cùng snapshot.

In [ ]:
assert digest(read(ROOT / "config/selected_retrieval.json")) == frozen_hash
with connect() as fresh_conn:
    fresh = Retriever(fresh_conn, embedder)
    assert fresh.qa_policy["selected_variant"] == chosen_name
    for case in qa_dev[:5] + qa_test[:5]:
        with capture_output():
            expected = runtime.qa_search(
                case["query"], policy_override=selected_policy, include_package=False
            )
            actual = fresh.qa_search(case["query"], include_package=False)
        assert [r["id"] for r in expected["matches"]] == [r["id"] for r in actual["matches"]]
print("QA runtime parity: 10/10 exact Top-10.")
assert {k: v for k, v in read(ROOT / "config/selected_retrieval.json").items() if k != "qa"} == {
    k: v for k, v in hs_selection.items() if k != "qa"
}
print("HS configuration unchanged by QA selection.")

In [ ]:
import subprocess

validation_env = {**os.environ, "RAG_RUN_DB_TESTS": "1", "TOKENIZERS_PARALLELISM": "false"}
for command in [
    [str(ROOT / ".venv/bin/ruff"), "check", "rag", "tests", "notebooks"],
    [str(ROOT / ".venv/bin/mypy"), "rag"],
    [str(ROOT / ".venv/bin/bandit"), "-r", "rag", "-ll", "-q"],
    [str(ROOT / ".venv/bin/python"), "-m", "pytest", "tests", "-q"],
    ["git", "diff", "--check"],
]:
    checked = subprocess.run(
        command, cwd=ROOT, env=validation_env, capture_output=True, text=True, check=False
    )
    print(
        " ".join(command),
        "exit",
        checked.returncode,
        checked.stdout[-700:] or checked.stderr[-700:] or "PASS",
    )
    assert checked.returncode == 0
selection_now = read(ROOT / "config/selected_retrieval.json")
assert selection_now["snapshot"] == runtime.snapshot
if "frozen_hash" in globals():
    assert digest(selection_now) == frozen_hash
print("Completed: snapshot", bundle["snapshot"], "QA policy", selection_now["qa"]["selected_variant"])

## QA regression on the new active Knowledge Base snapshot

The HS corpus was refreshed from the verified source. Recheck the already selected QA policy on all 150 supplied QA test questions without changing its weights or the frozen HS policy. These are retrieval metrics, not legal conclusions.

In [ ]:
import numpy as np

from rag.evaluation import qa_coverage

policy_hash = digest(read(ROOT / "config/selected_retrieval.json"))
runtime = Retriever(conn, embedder)
assert runtime.qa_policy["selected_variant"] == chosen_name
assert runtime.snapshot == read(ROOT / "config/selected_retrieval.json")["snapshot"]
qa_test_current = discover_qa(bundle, split="test")
assert len(qa_test_current) == 150
qa_rows = []
for index, case in enumerate(qa_test_current, 1):
    with capture_output() as progress:
        result = runtime.qa_search(case["query"])
    direct = qa_direct_coverage(case, result["matches"])
    package = qa_coverage(case, result["matches"], result["package"])
    qa_rows.append({"id": case["id"], "category": case["category"],
        "direct_all@5": direct["all_groups@5"],
        "direct_all@10": direct["all_groups@10"],
        "direct_group_recall@10": direct["group_recall@10"],
        "direct_mrr@10": direct["mrr@10"],
        "package_all": package["package_all_groups"],
        "package_group_recall": package["package_group_recall"],
        "missing_package": package["missing_package_groups"],
        "latency_ms": result["elapsed_ms"],
        "top10_ids": direct["returned_top10"]})
    if index % 25 == 0:
        print("QA regression", index, "/", len(qa_test_current), flush=True)
assert digest(read(ROOT / "config/selected_retrieval.json")) == policy_hash
frame = pd.DataFrame(qa_rows)
summary = frame.groupby("category").agg(
    questions=("id", "count"),
    direct_all_at_5=("direct_all@5", "mean"),
    direct_all_at_10=("direct_all@10", "mean"),
    direct_group_recall_at_10=("direct_group_recall@10", "mean"),
    direct_mrr_at_10=("direct_mrr@10", "mean"),
    package_all=("package_all", "mean"),
    latency_p50_ms=("latency_ms", lambda x: float(np.percentile(x, 50))),
    latency_p95_ms=("latency_ms", lambda x: float(np.percentile(x, 95))))
display(summary)
print("Overall direct all@10", frame["direct_all@10"].mean(),
      "package all", frame["package_all"].mean())
print("QA policy hash unchanged", policy_hash)
display(frame)

